In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
base_path = Path("data/concate_data")
green_path = base_path / "green_tripdata_2025_all.parquet"
yellow_path = base_path / "yellow_tripdata_2025_all.parquet"

green_df = pd.read_parquet(green_path)
yellow_df = pd.read_parquet(yellow_path)

print(f"Green shape: {green_df.shape}")
print(f"Yellow shape: {yellow_df.shape}")

Green shape: (543139, 21)
Yellow shape: (44417596, 20)


In [3]:
yellow_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [4]:
# latlon_terms = ("lat", "lon", "latitude", "longitude")
# for n, d in [("green", green_df), ("yellow", yellow_df)]:
#     cols = [c for c in d.columns if any(t in c.lower() for t in latlon_terms)]
#     print(f"{n}: {cols if cols else 'No latitude/longitude columns found'}")

In [5]:
# Quick EDA: concise structure, quality, and key numeric checks
for name, df in [("green", green_df), ("yellow", yellow_df)]:
    print(f"\n=== {name.upper()} DATASET ===")
    print("Rows, Cols:", df.shape)
    
    dtype_counts = df.dtypes.value_counts()
    print("Dtype counts:")
    print(dtype_counts.to_string())
    
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    print("\nTop 5 columns by missing %:")
    print(missing_pct.head(5).round(2).to_string())
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    key_numeric = [c for c in [
        "trip_distance", "fare_amount", "tip_amount", "total_amount",
        "passenger_count", "tolls_amount"
    ] if c in numeric_cols]
    
    if key_numeric:
        print("\nKey numeric summary:")
        print(df[key_numeric].describe().round(2).to_string())
    
    print("\nSample rows:")
    print(df.head(2).to_string(index=False))


=== GREEN DATASET ===
Rows, Cols: (543139, 21)
Dtype counts:
float64           15
int32              3
datetime64[us]     2
object             1

Top 5 columns by missing %:
ehail_fee             100.00
trip_type               8.12
passenger_count         8.11
store_and_fwd_flag      8.11
RatecodeID              8.11

Key numeric summary:
       trip_distance  fare_amount  tip_amount  total_amount  passenger_count  tolls_amount
count      543139.00    543139.00   543139.00     543139.00        499102.00     543139.00
mean           18.43        18.18        2.67         25.23             1.29          0.27
std          1137.17        17.73        3.65         19.99             0.94          1.44
min             0.00      -470.60     -100.00       -473.10             0.00         -6.94
25%             1.21         9.30        0.00         14.64             1.00          0.00
50%             1.98        13.58        2.08         20.02             1.00          0.00
75%             3.51 

In [6]:
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_error

# Build yellow-only modeling frame and target duration in minutes
yellow_model_df = yellow_df.copy()
yellow_model_df["duration"] = (
    (yellow_model_df["tpep_dropoff_datetime"] - yellow_model_df["tpep_pickup_datetime"]).dt.total_seconds() / 60
).round(2)

print(yellow_model_df["duration"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9,0.95,0.99]).round(2))

# Keep only valid durations for training
yellow_model_df = yellow_model_df[(yellow_model_df["duration"] > 0) & (yellow_model_df["duration"] <= 70)].copy()

# Drop columns that directly leak target or are always null/highly problematic
drop_cols = [c for c in ["tpep_dropoff_datetime","store_and_fwd_flag"] if c in yellow_model_df.columns]
yellow_model_df = yellow_model_df.drop(columns=drop_cols)

# Quick-run sample so AutoGluon trains in reasonable time
sample_n = min(200000, len(yellow_model_df))
yellow_model_df = yellow_model_df.sample(n=sample_n, random_state=42).reset_index(drop=True)

print("Prepared yellow_model_df shape:", yellow_model_df.shape)
print("duration preview:")
print(yellow_model_df["duration"].describe().round(2))

c:\Users\Martin\anaconda3\envs\autogluon_win\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


count    44417596.00
mean           17.20
std            28.44
min        -51472.32
10%             4.83
25%             8.08
50%            13.38
75%            21.28
90%            32.33
95%            42.93
99%            69.90
max         14880.77
Name: duration, dtype: float64
Prepared yellow_model_df shape: (200000, 19)
duration preview:
count    200000.00
mean         16.34
std          11.67
min           0.02
25%           8.22
50%          13.40
75%          21.08
max          70.00
Name: duration, dtype: float64


In [7]:
# Train/test split (model fit will use train only)
train_df, test_df = train_test_split(yellow_model_df, test_size=0.2, random_state=42)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (160000, 19)
Test shape: (40000, 19)


In [ ]:
label = "duration"
predictor = TabularPredictor(
    label=label,
    problem_type="regression",
    eval_metric="mae",
    path="AutogluonModels/yellow_duration_quick"
)

# Fit only on train_df (AutoGluon internally creates validation data from train_df)
predictor.fit(
    train_data=train_df,
    presets=['medium_quality_faster_train', 'optimize_for_deployment'],
    time_limit=300,
    verbosity=2
)



Preset alias specified: 'medium_quality_faster_train' maps to 'medium_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.10.20
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       22.36 GB / 31.75 GB (70.4%)
Disk Space Avail:   141.96 GB / 441.83 GB (32.1%)
Presets specified: ['medium_quality_faster_train']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 300s
AutoGluon will save models to "c:\Users\Martin\Desktop\Gatech\6242\Project\CSE6242_Team82\AutogluonModels\yellow_duration_quick"
Train Data Rows:    160000
Train Data Columns: 18
Label Column:       duration
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGen

[1000]	valid_set's l1: 2.20412
[2000]	valid_set's l1: 2.14484
[3000]	valid_set's l1: 2.13046
[4000]	valid_set's l1: 2.12359
[5000]	valid_set's l1: 2.12033
[6000]	valid_set's l1: 2.1246


	-2.1194	 = Validation score   (-mean_absolute_error)
	12.87s	 = Training   runtime
	0.08s	 = Validation runtime
Fitting model: LightGBM ... Training model for up to 286.51s of the 286.50s of remaining time.
	Fitting with cpus=10, gpus=0, mem=0.1/26.4 GB


[1000]	valid_set's l1: 2.11828
[2000]	valid_set's l1: 2.09734
[3000]	valid_set's l1: 2.10023
[4000]	valid_set's l1: 2.09512


	-2.0927	 = Validation score   (-mean_absolute_error)
	6.96s	 = Training   runtime
	0.05s	 = Validation runtime
Fitting model: RandomForestMSE ... Training model for up to 279.34s of the 279.34s of remaining time.
	Fitting with cpus=16, gpus=0, mem=0.7/26.2 GB
	-2.1172	 = Validation score   (-mean_absolute_error)
	27.24s	 = Training   runtime
	0.07s	 = Validation runtime
Fitting model: CatBoost ... Training model for up to 251.70s of the 251.69s of remaining time.
	Fitting with cpus=10, gpus=0
	-2.0057	 = Validation score   (-mean_absolute_error)
	132.99s	 = Training   runtime
	0.01s	 = Validation runtime
Fitting model: ExtraTreesMSE ... Training model for up to 118.67s of the 118.67s of remaining time.
	Fitting with cpus=16, gpus=0, mem=0.7/25.4 GB
	-2.141	 = Validation score   (-mean_absolute_error)
	12.61s	 = Training   runtime
	0.07s	 = Validation runtime
Fitting model: NeuralNetFastAI ... Training model for up to 105.65s of the 105.65s of remaining time.
	Fitting with cpus=10, gpu

In [9]:
# Predict on holdout test split
test_pred = predictor.predict(test_df.drop(columns=[label]))
test_mae = mean_absolute_error(test_df[label], test_pred)
print(f"Holdout Test MAE (minutes): {test_mae:.3f}")

Holdout Test MAE (minutes): 1.888


In [10]:
predictor.evaluate(test_df)

{'mean_absolute_error': -1.8878419022879562,
 'root_mean_squared_error': np.float64(-3.738765190321202),
 'mean_squared_error': -13.978365148357533,
 'r2': 0.8982476575531015,
 'pearsonr': 0.9478436316283212,
 'median_absolute_error': -0.7095625543594362}

In [11]:
predictor.leaderboard(test_df)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-1.887842,-1.998648,mean_absolute_error,1.439201,0.127877,167.224867,0.005284,0.001398,0.040395,2,True,7
1,CatBoost,-1.904379,-2.005737,mean_absolute_error,0.076859,0.010545,132.988215,0.076859,0.010545,132.988215,1,True,4
2,LightGBM,-1.941814,-2.092709,mean_absolute_error,0.754340,0.049143,6.956112,0.754340,0.049143,6.956112,1,True,2
3,LightGBMXT,-1.979553,-2.119384,mean_absolute_error,1.329422,0.083531,12.874481,1.329422,0.083531,12.874481,1,True,1
4,RandomForestMSE,-1.988045,-2.117199,mean_absolute_error,0.602718,0.066791,27.240145,0.602718,0.066791,27.240145,1,True,3
5,ExtraTreesMSE,-1.994885,-2.141026,mean_absolute_error,0.583196,0.066666,12.608324,0.583196,0.066666,12.608324,1,True,5
6,NeuralNetFastAI,-3.927622,-3.992159,mean_absolute_error,0.386873,0.027789,105.483963,0.386873,0.027789,105.483963,1,True,6
